# Optimización del entrenaiento de Red Neuronal en Machine Learning aplicando el Método de Broyden (Cero Gradiente)

### Caso 3: Optimización Convexa de Pesos Sinápticos
* **Objetivo:** Minimizar la entropía cruzada resolviendo $\nabla \mathcal{L}(\mathbf{w}) = \mathbf{0}$.
* **Arquitectura:** Perceptrón simple de 2 entradas ($x_1, x_2$) hacia una salida sigmoide.
* **Contraste algorítmico:**
  * *Descenso de Gradiente:* Requiere cientos de épocas y calibrar manualmente el *learning rate*.
  * *Broyden:* Estima la curvatura (Hessiano) de forma implícita y converge en 4 a 5 iteraciones con búsqueda de línea.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ==========================================================
# 1. Dataset Sintético: Separación Izquierda / Derecha
# ==========================================================
np.random.seed(42)
# Clase 0 a la izquierda, Clase 1 a la derecha
clase_0 = np.random.randn(30, 2) * 0.7 + np.array([-1.8, 0.0])
clase_1 = np.random.randn(30, 2) * 0.7 + np.array([ 1.8, 0.0])

X = np.vstack([clase_0, clase_1])
y = np.hstack([np.zeros(30), np.ones(30)])

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20, 20)))

def gradiente_loss(w):
    pred = sigmoid(X @ w)
    return X.T @ (pred - y)

# ==========================================================
# 2. Broyden con Búsqueda de Línea (Descenso Estable)
# ==========================================================
def entrenar_broyden_estable(w_init, max_iter=7):
    w = np.array(w_init, dtype=float)
    historia_w = [w.copy()]
    historia_err = [np.linalg.norm(gradiente_loss(w))]
    B = np.eye(len(w)) * 5.0  # Escala inicial adecuada

    for _ in range(max_iter):
        g = gradiente_loss(w)
        norm_g = np.linalg.norm(g)
        if norm_g < 1e-3:
            break

        s = np.linalg.solve(B, -g)

        # Line search / amortiguación básica para evitar overshooting
        alpha = 1.0
        while np.linalg.norm(gradiente_loss(w + alpha * s)) > norm_g * 1.2 and alpha > 0.1:
            alpha *= 0.5
        s = alpha * s

        w_new = w + s
        y_diff = gradiente_loss(w_new) - g

        # Actualización de Broyden
        B = B + np.outer(y_diff - B @ s, s) / np.dot(s, s)

        w = w_new
        historia_w.append(w.copy())
        historia_err.append(np.linalg.norm(gradiente_loss(w)))

    return historia_w, historia_err

# Inicio en una frontera casi horizontal (clasificación pésima)
w_inicial = [0.15, 2.2]
historia_pesos, historia_error = entrenar_broyden_estable(w_inicial)

# ==========================================================
# 3. Animación en Dos Paneles
# ==========================================================
fig, (ax_net, ax_plot) = plt.subplots(1, 2, figsize=(11, 5))

pos_x1 = np.array([0.2, 0.75])
pos_x2 = np.array([0.2, 0.25])
pos_out = np.array([0.8, 0.5])

def dibujar_red_base():
    ax_net.clear()
    ax_net.set_xlim(-0.1, 1.1)
    ax_net.set_ylim(-0.1, 1.1)
    ax_net.axis('off')
    ax_net.set_title("Topología Neuronal (Pesos Sinápticos)", fontsize=11, fontweight='bold')

    for pos, label in [(pos_x1, r'$x_1$'), (pos_x2, r'$x_2$')]:
        ax_net.add_patch(plt.Circle(pos, 0.12, color='#2563eb', ec='black', lw=1.5, zorder=4))
        ax_net.text(pos[0], pos[1], label, ha='center', va='center', fontsize=12, color='white', fontweight='bold')

    ax_net.add_patch(plt.Circle(pos_out, 0.12, color='#10b981', ec='black', lw=1.5, zorder=4))
    ax_net.text(pos_out[0], pos_out[1], r'$\hat{y}$', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

x_vals = np.linspace(-3.5, 3.5, 100)

def update(frame):
    w = historia_pesos[frame]
    err = historia_error[frame]

    # 1. Red y pesos
    dibujar_red_base()
    c_w1 = '#2563eb' if w[0] >= 0 else '#dc2626'
    lw_w1 = np.clip(abs(w[0]) * 3.0, 1.5, 8.0)
    ax_net.plot([pos_x1[0], pos_out[0]], [pos_x1[1], pos_out[1]], color=c_w1, lw=lw_w1, zorder=2)
    ax_net.text(0.48, 0.68, f"w1 = {w[0]:.2f}", fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))

    c_w2 = '#2563eb' if w[1] >= 0 else '#dc2626'
    lw_w2 = np.clip(abs(w[1]) * 3.0, 1.5, 8.0)
    ax_net.plot([pos_x2[0], pos_out[0]], [pos_x2[1], pos_out[1]], color=c_w2, lw=lw_w2, zorder=2)
    ax_net.text(0.48, 0.30, f"w2 = {w[1]:.2f}", fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))

    # 2. Frontera en el plano
    ax_plot.clear()
    ax_plot.scatter(clase_0[:, 0], clase_0[:, 1], color='#ef4444', label='Clase 0', edgecolors='k', s=50)
    ax_plot.scatter(clase_1[:, 0], clase_1[:, 1], color='#10b981', label='Clase 1', edgecolors='k', s=50)

    # Ecuación: w1*x1 + w2*x2 = 0 => x2 = -(w1/w2)*x1
    if abs(w[1]) > 1e-4:
        y_line = -(w[0] / w[1]) * x_vals
        ax_plot.plot(x_vals, y_line, color='black', lw=2.5, linestyle='--', label='Frontera Broyden')
    else:
        ax_plot.axvline(0, color='black', lw=2.5, linestyle='--', label='Frontera Broyden')

    ax_plot.set_xlim(-3.5, 3.5)
    ax_plot.set_ylim(-3.5, 3.5)
    ax_plot.set_title(f"Paso {frame}: Error Gradiente ||F(w)|| = {err:.3e}", fontsize=11, fontweight='bold')
    ax_plot.grid(True, linestyle=':', alpha=0.6)
    ax_plot.legend(loc='upper left', fontsize=9)

anim = FuncAnimation(fig, update, frames=len(historia_pesos), interval=1000)
plt.close(fig)

HTML(anim.to_jshtml())